In [1]:
from __future__ import annotations

import base64
import io
import json
import logging
import os
import subprocess
import threading
import time
import yaml
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import cv2
import numpy as np
from langchain_core.tools import tool
from PIL import Image as PILImage

logger = logging.getLogger(__name__)

In [2]:
# ---------------------------------------------------------------------------
# Policy Registry
# ---------------------------------------------------------------------------

def _load_policies() -> dict[str, dict[str, str]]:
    policy_path = Path("/home/xle/agentic-xle/src/xlerobot_soft_orchestrator/policies.yaml")
    if not policy_path.exists():
        logger.warning("policies.yaml not found at %s. VLA policies unavailable.", policy_path)
        return {}
    with open(policy_path, "r") as f:
        return yaml.safe_load(f) or {}

POLICIES: dict[str, dict[str, str]] = _load_policies()
POLICIES

{'screw_picking': {'policy_type': 'smolvla',
  'repo_id': 'rlodhi/smolvla_screw_picking_clean_25k',
  'task': 'pick the screw'},
 'screw_placing': {'policy_type': 'smolvla',
  'repo_id': 'rlodhi/smolvla_screw_placing_5000',
  'task': 'place the screw in the container'}}

In [3]:
# ---------------------------------------------------------------------------
# VLA Process Management
# ---------------------------------------------------------------------------

_vla_process: subprocess.Popen | None = None

_VLA_SERVER         = os.getenv("VLA_INFERENCE_SERVER", "192.168.50.42:8080")
_ROBOT_PORT         = os.getenv("ROBOT_PORT", "/dev/ttyACM0")
_ROBOT_ID           = os.getenv("ROBOT_ID", "left_arm")
_VLA_START_TIMEOUT  = int(os.getenv("VLA_START_TIMEOUT", "300"))  # seconds

# Fires only after client.start() completes (model loaded on server) and both
# the action-receiver thread AND the control-loop thread have crossed their
# threading.Barrier — meaning the robot is about to send its first observation.
_VLA_READY_SIGNAL = "Action receiving thread starting"

ZMQ_HOST: str = os.getenv("ZMQ_SERVER_HOST", "localhost")
ZMQ_PORT: int = int(os.getenv("ZMQ_SERVER_PORT", "5555"))


def _drain_output(
    process: subprocess.Popen,
    ready_event: threading.Event,
    signal_seen: list[bool],
    output_buf: list[str],
) -> None:
    """Read merged stdout+stderr line-by-line, log at INFO, buffer for failure reports."""
    for raw in iter(process.stdout.readline, b""):
        line = raw.decode("utf-8", errors="replace").rstrip()
        if line:
            logger.info("[robot_client] %s", line)
            output_buf.append(line)
        if _VLA_READY_SIGNAL in line:
            signal_seen[0] = True
            ready_event.set()
    # EOF — process exited; unblock the waiter regardless
    ready_event.set()


def _start_vla_policy(policy_id: str, **kwargs: Any) -> dict[str, Any]:
    global _vla_process

    if _vla_process is not None and _vla_process.poll() is None:
        return {
            "status": "FAILURE",
            "error": "A VLA policy is already running. Call stop_vla_policy first.",
            "pid": _vla_process.pid,
        }

    policy_cfg = POLICIES.get(policy_id)
    if not policy_cfg:
        return {
            "status": "FAILURE",
            "error": (
                f"Policy '{policy_id}' not found in policies.yaml. "
                f"Available: {list(POLICIES.keys())}"
            ),
        }

    camera_config = json.dumps({
        "base": {
            "type": "zmq", "server_address": ZMQ_HOST, "port": ZMQ_PORT,
            "camera_name": "base", "width": 640, "height": 480, "fps": 30,
        },
        "wrist": {
            "type": "zmq", "server_address": ZMQ_HOST, "port": ZMQ_PORT,
            "camera_name": "wrist", "width": 640, "height": 480, "fps": 30,
        },
    })

    cmd = [
        "python", "-u", "-m", "lerobot.async_inference.robot_client",
        "--robot.type=so101_follower",
        f"--robot.port={_ROBOT_PORT}",
        f"--robot.id={_ROBOT_ID}",
        f"--robot.cameras={camera_config}",
        f"--task={policy_cfg['task']}",
        f"--server_address={_VLA_SERVER}",
        f"--policy_type={policy_cfg['policy_type']}",
        f"--pretrained_name_or_path={policy_cfg['repo_id']}",
        "--policy_device=cuda",
        "--client_device=cpu",
        "--actions_per_chunk=50",
    ]

    try:
        _vla_process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,  # merge stderr into stdout so we capture everything
        )
    except Exception as exc:
        return {"status": "FAILURE", "error": str(exc)}

    ready_event = threading.Event()
    signal_seen = [False]
    output_buf: list[str] = []

    threading.Thread(
        target=_drain_output,
        args=(_vla_process, ready_event, signal_seen, output_buf),
        daemon=True,
    ).start()

    logger.info(
        "Waiting for policy '%s' to load (timeout: %ds) — watch logs above for progress...",
        policy_id, _VLA_START_TIMEOUT,
    )
    ready_event.wait(timeout=_VLA_START_TIMEOUT)

    def _tail(n: int = 30) -> str:
        return "\n".join(output_buf[-n:]) if output_buf else "(no output captured)"

    if not signal_seen[0]:
        rc = _vla_process.poll()
        if rc is not None:
            err = (
                f"Policy '{policy_id}' process exited with code {rc} "
                "before inference started."
            )
        else:
            _vla_process.terminate()
            try:
                _vla_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                _vla_process.kill()
            err = (
                f"Policy '{policy_id}' did not start within {_VLA_START_TIMEOUT}s. "
                "Model download may have stalled or the gRPC server is unreachable."
            )
        _vla_process = None
        return {"status": "FAILURE", "error": err, "output_tail": _tail()}

    if _vla_process.poll() is not None:
        _vla_process = None
        return {
            "status": "FAILURE",
            "error": f"Policy '{policy_id}' exited immediately after starting inference.",
            "output_tail": _tail(),
        }

    return {
        "status": "SUCCESS",
        "message": (
            f"Policy '{policy_id}' is running — model loaded, action chunks generating "
            f"(task: \"{policy_cfg['task']}\", type: {policy_cfg['policy_type']})."
        ),
        "pid": _vla_process.pid,
    }


def _stop_vla_policy(**kwargs: Any) -> dict[str, Any]:
    global _vla_process
    if _vla_process is None or _vla_process.poll() is not None:
        return {"status": "WARNING", "message": "No VLA policy was running."}
    _vla_process.terminate()
    try:
        _vla_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        _vla_process.kill()
        _vla_process.wait()
    _vla_process = None
    return {"status": "SUCCESS", "message": "VLA policy stopped."}

In [6]:
_start_vla_policy("screw_picking")

{'status': 'FAILURE',
 'error': "Policy 'screw_picking' did not start within 300s. Model download may have stalled or the gRPC server is unreachable.",
 'output_tail': " 'policy_type': 'smolvla',\n 'pretrained_name_or_path': 'rlodhi/smolvla_screw_picking_clean_25k',\n 'robot': {'calibration_dir': None,\n           'cameras': {'base': {'camera_name': 'base',\n                                'color_mode': <ColorMode.RGB: 'rgb'>,\n                                'fps': 30,\n                                'height': 480,\n                                'port': 5555,\n                                'server_address': 'localhost',\n                                'timeout_ms': 5000,\n                                'warmup_s': 1,\n                                'width': 640},\n                       'wrist': {'camera_name': 'wrist',\n                                 'color_mode': <ColorMode.RGB: 'rgb'>,\n                                 'fps': 30,\n                                 'height'

# Diagnosis

The `output_tail` from the timeout shows exactly what happened — the subprocess **did** start successfully, but it blocked on this prompt from `_follower.py`:

```
Press ENTER to use provided calibration file associated with the id left_arm,
or type 'c' and press ENTER to run calibration:
```

The robot's `__init__` calls `input()` to confirm the calibration file. With no terminal attached to the subprocess, it hangs forever → 300s timeout.

**Fix**: open `stdin=subprocess.PIPE` and immediately write `\n` to auto-accept the existing calibration file.

In [ ]:
# Confirm diagnosis: run the subprocess with stdin=PIPE and send \n immediately.
# Watch the output — it should move past the calibration prompt and start connecting.

import subprocess, threading, time

cmd = [
    "python", "-u", "-m", "lerobot.async_inference.robot_client",
    "--robot.type=so101_follower",
    "--robot.port=/dev/ttyACM0",
    "--robot.id=left_arm",
    "--robot.cameras={}",          # empty cameras just to get past arg parsing quickly
    "--task=pick the screw",
    "--server_address=192.168.50.42:8080",
    "--policy_type=smolvla",
    "--pretrained_name_or_path=rlodhi/smolvla_screw_picking_clean_25k",
    "--policy_device=cuda",
    "--client_device=cpu",
    "--actions_per_chunk=50",
]

proc = subprocess.Popen(
    cmd,
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Send ENTER immediately to accept existing calibration file
proc.stdin.write(b"\n")
proc.stdin.flush()

# Collect output for 15 seconds so we can see what happens next
lines = []
stop = threading.Event()

def _read():
    for raw in iter(proc.stdout.readline, b""):
        line = raw.decode("utf-8", errors="replace").rstrip()
        lines.append(line)
        print(line)
        if stop.is_set():
            break

t = threading.Thread(target=_read, daemon=True)
t.start()
time.sleep(15)
stop.set()
proc.terminate()
print("\n--- done sampling ---")
print(f"Process exit code: {proc.poll()}")

## If the cell above moves past the calibration prompt → fix confirmed

The updated `_start_vla_policy` below adds `stdin=subprocess.PIPE` and sends `\n` right after `Popen`. Run this to do a full end-to-end test with the real timeout logic.

In [ ]:
def _start_vla_policy_fixed(policy_id: str) -> dict:
    global _vla_process

    if _vla_process is not None and _vla_process.poll() is None:
        return {"status": "FAILURE", "error": "Already running.", "pid": _vla_process.pid}

    policy_cfg = POLICIES.get(policy_id)
    if not policy_cfg:
        return {"status": "FAILURE", "error": f"Unknown policy '{policy_id}'. Available: {list(POLICIES.keys())}"}

    camera_config = json.dumps({
        "base":  {"type": "zmq", "server_address": ZMQ_HOST, "port": ZMQ_PORT,
                  "camera_name": "base",  "width": 640, "height": 480, "fps": 30},
        "wrist": {"type": "zmq", "server_address": ZMQ_HOST, "port": ZMQ_PORT,
                  "camera_name": "wrist", "width": 640, "height": 480, "fps": 30},
    })

    cmd = [
        "python", "-u", "-m", "lerobot.async_inference.robot_client",
        "--robot.type=so101_follower",
        f"--robot.port={_ROBOT_PORT}",
        f"--robot.id={_ROBOT_ID}",
        f"--robot.cameras={camera_config}",
        f"--task={policy_cfg['task']}",
        f"--server_address={_VLA_SERVER}",
        f"--policy_type={policy_cfg['policy_type']}",
        f"--pretrained_name_or_path={policy_cfg['repo_id']}",
        "--policy_device=cuda",
        "--client_device=cpu",
        "--actions_per_chunk=50",
    ]

    _vla_process = subprocess.Popen(
        cmd,
        stdin=subprocess.PIPE,   # KEY FIX: allows us to send \n past the calibration prompt
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    # Accept the existing calibration file automatically
    _vla_process.stdin.write(b"\n")
    _vla_process.stdin.flush()

    ready_event = threading.Event()
    signal_seen = [False]
    output_buf = []

    def _drain(proc, ev, seen, buf):
        for raw in iter(proc.stdout.readline, b""):
            line = raw.decode("utf-8", errors="replace").rstrip()
            if line:
                print(f"[robot_client] {line}")
                buf.append(line)
            if "Action receiving thread starting" in line:
                seen[0] = True
                ev.set()
        ev.set()

    threading.Thread(target=_drain, args=(_vla_process, ready_event, signal_seen, output_buf), daemon=True).start()

    print(f"Waiting for '{policy_id}' to load (timeout: {_VLA_START_TIMEOUT}s)...")
    ready_event.wait(timeout=_VLA_START_TIMEOUT)

    def tail(n=30): return "\n".join(output_buf[-n:]) if output_buf else "(no output)"

    if not signal_seen[0]:
        rc = _vla_process.poll()
        err = (f"Exited with code {rc} before inference started." if rc is not None
               else f"Timed out after {_VLA_START_TIMEOUT}s.")
        _vla_process = None
        return {"status": "FAILURE", "error": err, "output_tail": tail()}

    if _vla_process.poll() is not None:
        _vla_process = None
        return {"status": "FAILURE", "error": "Exited immediately after start.", "output_tail": tail()}

    return {
        "status": "SUCCESS",
        "message": f"Policy '{policy_id}' running — model loaded, chunks generating.",
        "pid": _vla_process.pid,
    }

result = _start_vla_policy_fixed("screw_picking")
print("\nResult:", result)

## Why `stop_vla_policy` didn't actually stop the robot

`subprocess.Popen` without `start_new_session=True` puts the child in the **same process group** as the orchestrator (Streamlit). When we call `_vla_process.terminate()` we only send SIGTERM to the top-level Python process. The gRPC client threads and motor driver threads are *inside* that process and should die with it — but if the Python process doesn't exit cleanly (e.g. a non-daemon thread holds a lock), SIGTERM is insufficient and the process lingers as an orphan once Streamlit dies.

**Fix**:
- `start_new_session=True` in `Popen` → subprocess gets its own session & process group
- `os.killpg(pgid, SIGTERM)` in `stop_vla_policy` → kills the entire group atomically

Test that `stop_vla_policy` actually works after the fix:

In [4]:
import os, signal, subprocess, time

# Start a long-running process in its own session
p = subprocess.Popen(
    ["sleep", "120"],
    start_new_session=True,
)
pid = p.pid
pgid = os.getpgid(pid)
print(f"Launched PID {pid}, PGID {pgid}")

time.sleep(1)
print(f"Process alive: {p.poll() is None}")  # should be True

# Kill the whole group
os.killpg(pgid, signal.SIGTERM)
p.wait(timeout=3)
print(f"Process alive after killpg: {p.poll() is None}")  # should be False
print(f"Exit code: {p.returncode}")  # should be -15 (SIGTERM)

Launched PID 11697, PGID 11697
Process alive: True
Process alive after killpg: False
Exit code: -15
